# Surrogate 資料產線（Freerouting 標註）

**CPU runtime**。每 netlist → SA anchor → 8 品質變體 → 各跑 Freerouting 拿 routed_fraction →
存 (graph/raster, routed_fraction) 到 Drive。**斷點續跑**：session 斷了重跑會接續。

> 先跑小批（N_NETLISTS≈20，~1hr）驗證訊號，再放大。標註慢（每板 ~20-30s）。
> Secrets：`GH_TOKEN`。

## 1. 裝 KiCad / Java 25 / Freerouting / xvfb

In [ ]:
!apt-get -qq install -y software-properties-common xvfb > /dev/null
!add-apt-repository -y ppa:kicad/kicad-9.0-releases > /dev/null 2>&1
!apt-get -qq update > /dev/null
!apt-get -qq install -y --no-install-recommends kicad > /dev/null
# Temurin 25（最新 Freerouting 需 Java 25）
!wget -qO - https://packages.adoptium.net/artifactory/api/gpg/key/public | gpg --dearmor | tee /etc/apt/trusted.gpg.d/adoptium.gpg > /dev/null
!echo "deb https://packages.adoptium.net/artifactory/deb $(lsb_release -cs) main" | tee /etc/apt/sources.list.d/adoptium.list > /dev/null
!apt-get -qq update > /dev/null
!apt-get -qq install -y temurin-25-jdk > /dev/null
import json, urllib.request
rel = json.load(urllib.request.urlopen("https://api.github.com/repos/freerouting/freerouting/releases/latest"))
urllib.request.urlretrieve(next(a["browser_download_url"] for a in rel["assets"] if a["name"].endswith(".jar")), "/content/freerouting.jar")
!java -version

## 2. repo + Drive

In [ ]:
import os, sys
from google.colab import drive, userdata
GH = userdata.get("GH_TOKEN")
if not os.path.isdir("/content/genpcb"):
    !git clone https://{GH}@github.com/timmytsaa/genpcb.git /content/genpcb
else:
    !git -C /content/genpcb pull
!pip -q install -e /content/genpcb
sys.path.insert(0, "/content/genpcb/src")
drive.mount("/content/drive")

## 3. 設定

In [ ]:
N_NETLISTS = 20      # 先小批驗證；放大就調這個重跑（斷點續跑接續）
MAX_PASSES = 5
OUT_DIR = "/content/drive/MyDrive/genpcb/surrogate_data"

## 4. 跑標註（慢；斷線重跑此格會接續）

In [ ]:
import time
from genpcb.kicad.route import routing_reward
from genpcb.surrogate.gen_data import build_dataset

t = time.time()
man = build_dataset(
    N_NETLISTS, OUT_DIR,
    label_fn=lambda b: routing_reward(b, jar="/content/freerouting.jar", max_passes=MAX_PASSES),
    sa_steps=2500,
)
print(f"{len(man)} samples ({len(man)//8} netlists) in {(time.time()-t)/60:.1f} min")

## 5. 驗證訊號（放大前先看這個）
routed_fraction 要有跨度，且 **base（SA anchor）平均 > scatter100** = 訊號反映擺位品質。

In [ ]:
import json, numpy as np
from collections import defaultdict
rows = [json.loads(l) for l in open(OUT_DIR + "/manifest.jsonl")]
rf = [r["routed_fraction"] for r in rows if r["routed_fraction"] is not None]
print(f"labeled {len(rf)}/{len(rows)} | routed_fraction min {min(rf):.2f} mean {np.mean(rf):.2f} max {max(rf):.2f}")
by = defaultdict(list)
for r in rows:
    if r["routed_fraction"] is not None:
        by[r["variant"]].append(r["routed_fraction"])
for v in ["base", "jitter1", "jitter5", "jitter20", "scatter25", "scatter50", "scatter100", "swap3"]:
    if by[v]:
        print(f"  {v:11s} mean routed_fraction {np.mean(by[v]):.3f}  (n={len(by[v])})")
print("\n訊號 OK 條件：base 明顯 > scatter100，且整體有跨度")